In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from lib.dgm import Net_DGM
from lib.networks import FFN
from EX1_1 import LQR


# DEVICE / SEED

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

torch.manual_seed(0)



def lqr_value_batch(lqr, t, x, out_device):
    """
    Adapt current code shapes to EX1_1_xinyue.LQR.value_function

    Inputs
    ------
    t : (batch, 1) or (batch,)
    x : (batch, 2)

    Returns
    -------
    value : (batch, 1)
    """
    if t.dim() == 2:
        t_in = t.squeeze(1)
    else:
        t_in = t

    x_in = x.unsqueeze(1)  # (batch, 1, 2)

    # LQR in EX1_1_xinyue is safest on CPU
    val = lqr.value_function(t_in.detach().cpu(), x_in.detach().cpu())
    return val.to(out_device)


def lqr_control_batch(lqr, t, x, out_device):
    """
    Adapt current code shapes to EX1_1.LQR.optimal_control
    Inputs
    t : (batch, 1) or (batch,)
    x : (batch, 2)

    Returns
    control : (batch, 2)
    """
    if t.dim() == 2:
        t_in = t.squeeze(1)
    else:
        t_in = t

    x_in = x.unsqueeze(1)  # (batch, 1, 2)

    ctrl = lqr.optimal_control(t_in.detach().cpu(), x_in.detach().cpu())
    return ctrl.to(out_device)


# 2. SECOND-ORDER TERM FOR DGM CRITIC
# critic = Net_DGM(t, x)

def diffusion_term_from_x(v, x, sigma):
    """
    return 0.5 * tr(sigma sigma^T Hess_x v)
    """
    vx = torch.autograd.grad(
        v, x,
        grad_outputs=torch.ones_like(v),
        create_graph=True,
        retain_graph=True
    )[0]

    h_rows = []
    for i in range(2):
        grad_i = vx[:, i:i+1]
        second = torch.autograd.grad(
            grad_i, x,
            grad_outputs=torch.ones_like(grad_i),
            create_graph=True,
            retain_graph=True
        )[0]
        h_rows.append(second)

    Hx = torch.stack(h_rows, dim=1)  # (batch, 2, 2)
    A = sigma @ sigma.T
    return 0.5 * torch.einsum('ij,bij->b', A, Hx).reshape(-1, 1)

In [ ]:

# 3. PIA object
# critic minimizes:
# R(theta) = R_eqn(theta) + R_boundary(theta)
# actor minimizes Hamiltonian

class LQR_PIA:
    def __init__(self, critic, actor, H, M, C, D, R, sigma, T, device):
        self.critic = critic   # Net_DGM
        self.actor = actor     # FFN
        self.H = H
        self.M = M
        self.C = C
        self.D = D
        self.R = R
        self.sigma = sigma
        self.T = T
        self.device = device

    def pde_residual(self, size):
        t = torch.rand(size, 1, device=self.device) * self.T
        x = -2 + 4 * torch.rand(size, 2, device=self.device)

        t = t.clone().detach().requires_grad_(True)
        x = x.clone().detach().requires_grad_(True)

        tx = torch.cat([t, x], dim=1)

        # critic is DGM: critic(t, x)
        v = self.critic(t, x)

        vt = torch.autograd.grad(
            v, t,
            grad_outputs=torch.ones_like(v),
            create_graph=True,
            retain_graph=True
        )[0]

        vx = torch.autograd.grad(
            v, x,
            grad_outputs=torch.ones_like(v),
            create_graph=True,
            retain_graph=True
        )[0]

        a = self.actor(tx)

        drift = x @ self.H.T + a @ self.M.T
        diff = diffusion_term_from_x(v, x, self.sigma)

        xCx = torch.sum((x @ self.C) * x, dim=1, keepdim=True)
        aDa = torch.sum((a @ self.D) * a, dim=1, keepdim=True)

        residual = vt + diff + torch.sum(vx * drift, dim=1, keepdim=True) + xCx + aDa
        return residual

    def terminal_residual(self, size):
        x = -2 + 4 * torch.rand(size, 2, device=self.device)
        t = torch.ones(size, 1, device=self.device) * self.T

        vT = self.critic(t, x)
        target = torch.sum((x @ self.R) * x, dim=1, keepdim=True)
        return vT - target

    def value_objective(self, size=2**8, return_parts=False):
        """
        R(theta) = R_eqn(theta) + R_boundary(theta)
        """
        eqn_res = self.pde_residual(size)
        bdry_res = self.terminal_residual(size)

        R_eqn = torch.mean(eqn_res ** 2)
        R_boundary = torch.mean(bdry_res ** 2)
        R_total = R_eqn + R_boundary

        if return_parts:
            return R_total, R_eqn, R_boundary
        return R_total

    def hamiltonian(self, size=2**8):
        t = torch.rand(size, 1, device=self.device) * self.T
        x = -2 + 4 * torch.rand(size, 2, device=self.device)

        t = t.clone().detach().requires_grad_(True)
        x = x.clone().detach().requires_grad_(True)

        tx = torch.cat([t, x], dim=1)

        v = self.critic(t, x)

        vx = torch.autograd.grad(
            v, x,
            grad_outputs=torch.ones_like(v),
            create_graph=True,
            retain_graph=True
        )[0]

        a = self.actor(tx)

        Hx = x @ self.H.T
        Ma = a @ self.M.T

        Hamil = (
            torch.sum(vx * Hx, dim=1, keepdim=True)
            + torch.sum(vx * Ma, dim=1, keepdim=True)
            + torch.sum((x @ self.C) * x, dim=1, keepdim=True)
            + torch.sum((a @ self.D) * a, dim=1, keepdim=True)
        )
        return torch.mean(Hamil)



# 4. TRAINER

class TrainPIA:
    def __init__(self, critic, actor, pia, lqr, device):
        self.critic = critic
        self.actor = actor
        self.pia = pia
        self.lqr = lqr
        self.device = device

        # outer-iteration summaries
        self.outer_idx = []
        self.min_total_obj_list = []
        self.min_eqn_obj_list = []
        self.min_boundary_obj_list = []
        self.min_hamiltonian_list = []

        self.value_abs_error_list = []
        self.action_abs_error_list = []

        # epoch-level histories for DGM-style plots
        self.value_epoch_hist = []
        self.total_loss_hist = []
        self.pde_loss_hist = []
        self.terminal_loss_hist = []

        self.error_eval_epoch_hist = []
        self.value_mae_hist = []

    def evaluate_absolute_errors(self, n_test=300):
        x = -2 + 4 * torch.rand(n_test, 2, device=self.device)
        t = torch.zeros(n_test, 1, device=self.device)
        tx = torch.cat([t, x], dim=1)

        with torch.no_grad():
            v_true = lqr_value_batch(self.lqr, t, x, self.device)
            a_true = lqr_control_batch(self.lqr, t, x, self.device)

            v_nn = self.critic(t, x)
            a_nn = self.actor(tx)

            value_abs_error = torch.mean(torch.abs(v_nn - v_true)).item()
            action_abs_error = torch.mean(torch.abs(a_nn - a_true)).item()

        return value_abs_error, action_abs_error

    def compute_value_mae(self, n_test=400):
        x = -2 + 4 * torch.rand(n_test, 2, device=self.device)
        t = torch.rand(n_test, 1, device=self.device) * self.pia.T

        with torch.no_grad():
            v_nn = self.critic(t, x)
            v_true = lqr_value_batch(self.lqr, t, x, self.device)
            mae = torch.mean(torch.abs(v_nn - v_true)).item()

        return mae

    def train(self, outer_iters=8, value_steps=3000, policy_steps=1500,
              lr_v=0.002, lr_a=0.001, eval_every=250):
        opt_v = optim.Adam(self.critic.parameters(), lr=lr_v)
        opt_a = optim.Adam(self.actor.parameters(), lr=lr_a)

        global_value_epoch = 0

        for k in range(outer_iters):
            print(f"\n=== Outer Iteration {k+1}/{outer_iters} ===")

        
            # critic update
            self.actor.eval()
            self.critic.train()

            min_total_obj = float("inf")
            min_eqn_obj = float("inf")
            min_boundary_obj = float("inf")

            for e in range(value_steps):
                global_value_epoch += 1

                opt_v.zero_grad()
                total_obj, eqn_obj, boundary_obj = self.pia.value_objective(size=2**8, return_parts=True)
                total_obj.backward()
                opt_v.step()

                min_total_obj = min(min_total_obj, total_obj.item())
                min_eqn_obj = min(min_eqn_obj, eqn_obj.item())
                min_boundary_obj = min(min_boundary_obj, boundary_obj.item())

                self.value_epoch_hist.append(global_value_epoch)
                self.total_loss_hist.append(total_obj.item())
                self.pde_loss_hist.append(eqn_obj.item())
                self.terminal_loss_hist.append(boundary_obj.item())

                if global_value_epoch % eval_every == 0:
                    value_mae = self.compute_value_mae(n_test=400)
                    self.error_eval_epoch_hist.append(global_value_epoch)
                    self.value_mae_hist.append(value_mae)

                if e % 300 == 299:
                    print(
                        f"Value step {e+1} | "
                        f"R_total = {total_obj.item():.6f}, "
                        f"R_eqn = {eqn_obj.item():.6f}, "
                        f"R_boundary = {boundary_obj.item():.6f}"
                    )

            # actor update
            self.critic.eval()
            for p in self.critic.parameters():
                p.requires_grad = False

            self.actor.train()
            min_hamiltonian = float("inf")

            for e in range(policy_steps):
                opt_a.zero_grad()
                hamil = self.pia.hamiltonian(size=2**8)
                hamil.backward()
                opt_a.step()

                min_hamiltonian = min(min_hamiltonian, hamil.item())

                if e % 300 == 299:
                    print(f"Policy step {e+1} | Hamiltonian = {hamil.item():.6f}")

            for p in self.critic.parameters():
                p.requires_grad = True

            
            # outer summary
            value_abs_error, action_abs_error = self.evaluate_absolute_errors()

            self.outer_idx.append(k + 1)
            self.min_total_obj_list.append(min_total_obj)
            self.min_eqn_obj_list.append(min_eqn_obj)
            self.min_boundary_obj_list.append(min_boundary_obj)
            self.min_hamiltonian_list.append(min_hamiltonian)
            self.value_abs_error_list.append(value_abs_error)
            self.action_abs_error_list.append(action_abs_error)

            print(
                f"Outer {k+1} summary | "
                f"min R_total = {min_total_obj:.6e}, "
                f"min R_eqn = {min_eqn_obj:.6e}, "
                f"min R_boundary = {min_boundary_obj:.6e}, "
                f"min Hamiltonian = {min_hamiltonian:.6e}, "
                f"value abs error = {value_abs_error:.6e}, "
                f"action abs error = {action_abs_error:.6e}"
            )

In [ ]:

# 5. STANDARD FIGURES
def plot_all_results(critic, actor, lqr, trainer, device):
    x1_grid = torch.linspace(-2, 2, 400, device=device).reshape(-1, 1)
    x2_grid = torch.zeros_like(x1_grid)
    t_grid = torch.zeros_like(x1_grid)

    x = torch.cat([x1_grid, x2_grid], dim=1)
    tx = torch.cat([t_grid, x1_grid, x2_grid], dim=1)

    with torch.no_grad():
        v_nn = critic(t_grid, x).squeeze(1).cpu()
        a_nn = actor(tx).cpu()
        v_true = lqr_value_batch(lqr, t_grid, x, device).squeeze(1).cpu()
        a_true = lqr_control_batch(lqr, t_grid, x, device).cpu()

    v_abs_err = torch.abs(v_nn - v_true)
    a1_abs_err = torch.abs(a_nn[:, 0] - a_true[:, 0])
    a2_abs_err = torch.abs(a_nn[:, 1] - a_true[:, 1])

    x1_list = x1_grid.squeeze(1).cpu().tolist()

    # Figure 1
    fig, axes = plt.subplots(
        2, 1, figsize=(7, 6),
        gridspec_kw={'height_ratios': [3, 1]},
        sharex=True
    )

    axes[0].plot(x1_list, v_true.tolist(), label='Exact value', linewidth=2)
    axes[0].plot(x1_list, v_nn.tolist(), '--', label='Learned value', linewidth=2)
    axes[0].set_title('Value Function Comparison')
    axes[0].set_ylabel(r'$v(0, x_1, 0)$')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(x1_list, v_abs_err.tolist(), linewidth=2)
    axes[1].set_xlabel(r'$x_1$  (with $t=0,\ x_2=0$)')
    axes[1].set_ylabel(r'$|error|$')
    axes[1].set_title('Absolute Error in Value Function')
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig("fig1_value_comparison.png", dpi=220, bbox_inches='tight')
    plt.close()

    # Figure 2
    fig, axes = plt.subplots(
        2, 1, figsize=(7, 7),
        gridspec_kw={'height_ratios': [3, 1.4]},
        sharex=True
    )

    axes[0].plot(x1_list, a_true[:, 0].tolist(), label='Exact $a_1$', linewidth=2)
    axes[0].plot(x1_list, a_nn[:, 0].tolist(), '--', label='Learned $a_1$', linewidth=2)
    axes[0].plot(x1_list, a_true[:, 1].tolist(), label='Exact $a_2$', linewidth=2)
    axes[0].plot(x1_list, a_nn[:, 1].tolist(), '--', label='Learned $a_2$', linewidth=2)
    axes[0].set_title('Control Comparison')
    axes[0].set_ylabel(r'$a(0, x_1, 0)$')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(x1_list, a1_abs_err.tolist(), label=r'$|a_1-a_1^*|$', linewidth=2)
    axes[1].plot(x1_list, a2_abs_err.tolist(), label=r'$|a_2-a_2^*|$', linewidth=2)
    axes[1].set_xlabel(r'$x_1$  (with $t=0,\ x_2=0$)')
    axes[1].set_ylabel(r'$|error|$')
    axes[1].set_title('Absolute Error in Control')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig("fig2_control_comparison.png", dpi=220, bbox_inches='tight')
    plt.close()

    # Figure 3
    fig, axes = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

    axes[0].plot(trainer.outer_idx, trainer.min_total_obj_list, marker='o', linewidth=2, label='min $R(\\theta)$')
    axes[0].plot(trainer.outer_idx, trainer.min_eqn_obj_list, marker='s', linewidth=2, label='min $R_{eqn}(\\theta)$')
    axes[0].plot(trainer.outer_idx, trainer.min_boundary_obj_list, marker='^', linewidth=2, label='min $R_{boundary}(\\theta)$')
    axes[0].set_yscale('log')
    axes[0].set_ylabel('Objective value')
    axes[0].set_title('Minimum Critic Objective Across Outer Iterations')
    axes[0].legend()
    axes[0].grid(True, which='both')

    axes[1].plot(trainer.outer_idx, trainer.min_hamiltonian_list, marker='o', linewidth=2, label='min Hamiltonian')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Outer iteration')
    axes[1].set_ylabel('Hamiltonian')
    axes[1].set_title('Minimum Hamiltonian Across Outer Iterations')
    axes[1].legend()
    axes[1].grid(True, which='both')

    plt.tight_layout()
    plt.savefig("fig3_minimum_objective_history.png", dpi=220, bbox_inches='tight')
    plt.close()

    print("Saved figures:")
    print(" - fig1_value_comparison.png")
    print(" - fig2_control_comparison.png")
    print(" - fig3_minimum_objective_history.png")
    print(f"Max abs value error         = {torch.max(v_abs_err).item():.6e}")
    print(f"Mean abs value error        = {torch.mean(v_abs_err).item():.6e}")
    print(f"Max abs control error (a1)  = {torch.max(a1_abs_err).item():.6e}")
    print(f"Mean abs control error (a1) = {torch.mean(a1_abs_err).item():.6e}")
    print(f"Max abs control error (a2)  = {torch.max(a2_abs_err).item():.6e}")
    print(f"Mean abs control error (a2) = {torch.mean(a2_abs_err).item():.6e}")

In [ ]:

# 6. DGM-STYLE FIGURES
def plot_dgm_style_results(trainer):
    # Figure 4: training loss
    plt.figure(figsize=(8, 5))
    plt.plot(trainer.value_epoch_hist, trainer.total_loss_hist, label='total loss')
    plt.plot(trainer.value_epoch_hist, trainer.pde_loss_hist, label='PDE loss')
    plt.plot(trainer.value_epoch_hist, trainer.terminal_loss_hist, label='terminal loss')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('EX4 PIA training loss')
    plt.legend()
    plt.grid(True, which='both')
    plt.tight_layout()
    plt.savefig("fig4_dgm_style_training_loss.png", dpi=220, bbox_inches='tight')
    plt.close()

    # Figure 5: error against Riccati benchmark
    plt.figure(figsize=(8, 5))
    plt.plot(trainer.error_eval_epoch_hist, trainer.value_mae_hist, marker='o')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Mean absolute error vs Riccati')
    plt.title('EX4 PIA error against Riccati benchmark')
    plt.grid(True, which='both')
    plt.tight_layout()
    plt.savefig("fig5_dgm_style_value_error.png", dpi=220, bbox_inches='tight')
    plt.close()

    print("Saved DGM-style figures:")
    print(" - fig4_dgm_style_training_loss.png")
    print(" - fig5_dgm_style_value_error.png")


In [ ]:

# 7. MAIN
if __name__ == "__main__":
    # problem parameters
    T = 1.0

    H = torch.tensor([[0.1, 0.0],
                      [0.0, 0.2]], dtype=torch.float32, device=device)

    M = torch.eye(2, dtype=torch.float32, device=device)
    C = torch.eye(2, dtype=torch.float32, device=device)
    D = torch.eye(2, dtype=torch.float32, device=device)
    R = torch.eye(2, dtype=torch.float32, device=device)

    sigma = 0.3 * torch.eye(2, dtype=torch.float32, device=device)

    # critic: DGM network for value v(t,x)
    critic = Net_DGM(dim_x=2, dim_S=128, activation='Tanh').to(device)

    # actor: FFN for control a(t,x)
    actor = FFN(
        sizes=[3, 64, 64, 2],
        activation=nn.Tanh,
        output_activation=nn.Identity,
        batch_norm=False
    ).to(device)

    # exact benchmark from Exercise 1.1
    lqr = LQR(H.cpu(), M.cpu(), C.cpu(), D.cpu(), R.cpu(), sigma.cpu(), T)
    t_grid = torch.linspace(0, T, 4000)
    lqr.solve_riccati(t_grid)

    # PIA
    pia = LQR_PIA(critic, actor, H, M, C, D, R, sigma, T, device)
    trainer = TrainPIA(critic, actor, pia, lqr, device)

    # train
    trainer.train(
    outer_iters=6,
    value_steps=1000,
    policy_steps=500,
    lr_v=0.002,
    lr_a=0.001,
    eval_every=125
)

    # standard figures
    plot_all_results(critic, actor, lqr, trainer, device)

    # DGM-style figures
    plot_dgm_style_results(trainer)


=== Outer Iteration 1/6 ===
Value step 300 | R_total = 0.005225, R_eqn = 0.003345, R_boundary = 0.001880
Value step 600 | R_total = 0.003135, R_eqn = 0.002169, R_boundary = 0.000966
Value step 900 | R_total = 0.001951, R_eqn = 0.001416, R_boundary = 0.000535
Policy step 300 | Hamiltonian = -3.737514
Outer 1 summary | min R_total = 7.305579e-04, min R_eqn = 4.366658e-04, min R_boundary = 1.028265e-04, min Hamiltonian = -4.949047e+00, value abs error = 4.527717e+00, action abs error = 1.326898e+00

=== Outer Iteration 2/6 ===
Value step 300 | R_total = 0.012190, R_eqn = 0.011323, R_boundary = 0.000867
Value step 600 | R_total = 0.004993, R_eqn = 0.004779, R_boundary = 0.000215
Value step 900 | R_total = 0.003384, R_eqn = 0.003200, R_boundary = 0.000184
Policy step 300 | Hamiltonian = 0.023455
Outer 2 summary | min R_total = 1.660817e-03, min R_eqn = 1.534558e-03, min R_boundary = 1.004817e-04, min Hamiltonian = -3.572897e-02, value abs error = 8.249025e-01, action abs error = 2.288218e-